<a href="https://colab.research.google.com/github/Malkavai-Misty/agentic-ai-demos/blob/main/notebooks/langgraph_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Cell 1: Install dependencies
!pip install -q langchain langchain-anthropic langgraph
print('Packages installed!')

Packages installed!


In [5]:
# Cell 2: Set API key — paste your key when prompted, then press Enter
import os
from getpass import getpass
os.environ['ANTHROPIC_API_KEY'] = getpass('Paste your Anthropic API key: ')
print('Key set!')

Paste your Anthropic API key: ··········
Key set!


In [6]:
# Cell 3: Define tools
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    weather_data = {
        "New York": "Partly cloudy, 68\u00b0F (20\u00b0C), humidity 65%",
        "Chicago": "Sunny, 72\u00b0F (22\u00b0C), humidity 50%",
        "LA": "Clear skies, 82\u00b0F (28\u00b0C), humidity 30%"
    }
    return weather_data.get(city, f"Weather data not available for {city}")

tools = [calculator, get_weather]
print('Tools defined!')

Tools defined!


In [7]:
# Cell 4: Build LangGraph ReAct agent
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
import operator

# Agent state
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

# Model bound to tools
model = ChatAnthropic(model='claude-haiku-4-5-20251001').bind_tools(tools)

# Node: think — LLM reasons and optionally requests tool calls
def think(state: AgentState) -> dict:
    response = model.invoke(state['messages'])
    return {'messages': [response]}

# Node: use_tool — executes any tool calls from the last message
def use_tool(state: AgentState) -> dict:
    return ToolNode(tools).invoke(state)

# Conditional edge: continue to tools or finish
def should_continue(state: AgentState) -> str:
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return 'use_tool'
    return END

# Build graph
graph = StateGraph(AgentState)
graph.add_node('think', think)
graph.add_node('use_tool', use_tool)
graph.set_entry_point('think')
graph.add_conditional_edges('think', should_continue)
graph.add_edge('use_tool', 'think')

agent = graph.compile()
print('Agent built successfully!')

Agent built successfully!


In [8]:
# Cell 5: Run the agent
from langchain_core.messages import HumanMessage

result = agent.invoke({
    'messages': [HumanMessage(content='What is 15% of 340? And what is the weather in Chicago and New York?')]
})

for msg in result['messages']:
    label = type(msg).__name__
    print(f'\n[{label}]')
    if hasattr(msg, 'content') and msg.content:
        print(msg.content)
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f'  -> Tool: {tc["name"]}  Args: {tc["args"]}')


[HumanMessage]
What is 15% of 340? And what is the weather in Chicago and New York?

[AIMessage]
[{'id': 'toolu_01R9RnDWyNHdGhQ6dNrM6iA6', 'caller': {'type': 'direct'}, 'input': {'expression': '0.15 * 340'}, 'name': 'calculator', 'type': 'tool_use'}, {'id': 'toolu_01VBchoa3njFo9MEtoCLAmpf', 'caller': {'type': 'direct'}, 'input': {'city': 'Chicago'}, 'name': 'get_weather', 'type': 'tool_use'}, {'id': 'toolu_01PENzpDhJhJKPtVMki6oBni', 'caller': {'type': 'direct'}, 'input': {'city': 'New York'}, 'name': 'get_weather', 'type': 'tool_use'}]
  -> Tool: calculator  Args: {'expression': '0.15 * 340'}
  -> Tool: get_weather  Args: {'city': 'Chicago'}
  -> Tool: get_weather  Args: {'city': 'New York'}

[ToolMessage]
51.0

[ToolMessage]
Sunny, 72°F (22°C), humidity 50%

[ToolMessage]
Partly cloudy, 68°F (20°C), humidity 65%

[AIMessage]
Here are the answers to your questions:

**15% of 340 = 51**

**Weather:**
- **Chicago**: Sunny, 72°F (22°C), humidity 50%
- **New York**: Partly cloudy, 68°F (2